# Decomposition Loss Weight Sweep

**Before running:** Runtime → Change runtime type → Hardware accelerator → **GPU**

Runs 12 decomposition experiments (small transformer h64/l3, 1000 epochs each):
- **Group `sweep_fft`**: FFT-only loss at weights 0.1 / 0.25 / 0.5 / 1.0
- **Group `sweep_trend`**: Trend-only loss at weights 0.1 / 0.25 / 0.5 / 1.0
- **Group `sweep_season`**: Season-only loss at weights 0.1 / 0.25 / 0.5 / 1.0

After each training run, full evaluation (disc + pred + VDS + FDDS + corr + **Context-FID**) is logged to W&B automatically.  
Each group of 4 runs is grouped together in W&B via the `group=` parameter — no manual configuration needed on the website.

## 1. Setup

In [ ]:
# ── Check GPU ────────────────────────────────────────────────────────────────
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory  : {mem_gb:.1f} GB")

In [ ]:
# ── Clone repo from GitHub ────────────────────────────────────────────────────
import os
REPO_DIR = '/content/DiffusionModelTimeSeries'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Ardameliksah/DiffusionModelTimeSeries.git {REPO_DIR}
else:
    print("Repo already cloned, pulling latest...")
    !git -C {REPO_DIR} pull

In [ ]:
import os, sys
from pathlib import Path

REPO_PATH = REPO_DIR
assert Path(REPO_PATH).exists(), f"Folder not found: {REPO_PATH}"

os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print(f"Working directory: {os.getcwd()}")
print("Files found:", [f for f in os.listdir() if f.endswith('.py')])

In [ ]:
# ── Install any missing packages ─────────────────────────────────────────────
!pip install -q --upgrade pip
!pip install -q wandb
!pip show tqdm scikit-learn seaborn | grep -E 'Name|Version'

In [ ]:
# ── Weights & Biases login ────────────────────────────────────────────────────
import wandb, os
os.environ["WANDB_API_KEY"] = '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'

_api_key = os.environ.get("WANDB_API_KEY")
if _api_key:
    wandb.login(key=_api_key, relogin=False)
else:
    wandb.login()

print("wandb version:", wandb.__version__)

## 2. Weight Sweep — 12 Decomposition Experiments

**W&B grouping** is controlled by the `group=` field in each experiment dict.  
In W&B → your project → click **Group** dropdown to browse by `sweep_fft` / `sweep_trend` / `sweep_season`.  
Each group contains both the training run and the post-training evaluation run.

**Timeline per experiment:** ~training time + ~2–3 min eval (incl. Context-FID TS2Vec training)

In [ ]:
import torch, gc, importlib
from pathlib import Path

import train_with_mode as _twm
importlib.reload(_twm)
from train_with_mode import train

from evaluate_unified import evaluate

# ── Shared settings ───────────────────────────────────────────────────────────
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
WANDB_PROJECT = "diffusion-timeseries"
NUM_WORKERS   = 2
SEED          = 42
BATCH_SIZE    = 64

# Small model config (all runs)
HIDDEN_DIM  = 64
NUM_LAYERS  = 3
NUM_EPOCHS  = 1000
LR          = 1e-4

# Inline metric settings — DiffusionTS protocol:
#   full iterations (disc=2000, pred=5000), 5 independent runs, full dataset
#   eval every 250 epochs → 4 checkpoints per 1000-epoch run
EVAL_METRICS_EVERY  = 250
N_METRIC_ITERATIONS = 5    # 5 independent runs (matches DiffusionTS)
# num_metric_samples no longer used — full dataset is always collected

# ── Experiment list ───────────────────────────────────────────────────────────
EXPERIMENTS = [

    # ── FFT loss sweep ────────────────────────────────────────────────────────
    dict(name="fft_w010", group="sweep_fft",
         fft_weight=0.10, trend_weight=0.0, season_weight=0.0),
    dict(name="fft_w025", group="sweep_fft",
         fft_weight=0.25, trend_weight=0.0, season_weight=0.0),
    dict(name="fft_w050", group="sweep_fft",
         fft_weight=0.50, trend_weight=0.0, season_weight=0.0),
    dict(name="fft_w100", group="sweep_fft",
         fft_weight=1.00, trend_weight=0.0, season_weight=0.0),

    # ── Trend loss sweep ──────────────────────────────────────────────────────
    dict(name="trend_w010", group="sweep_trend",
         fft_weight=0.0, trend_weight=0.10, season_weight=0.0),
    dict(name="trend_w025", group="sweep_trend",
         fft_weight=0.0, trend_weight=0.25, season_weight=0.0),
    dict(name="trend_w050", group="sweep_trend",
         fft_weight=0.0, trend_weight=0.50, season_weight=0.0),
    dict(name="trend_w100", group="sweep_trend",
         fft_weight=0.0, trend_weight=1.00, season_weight=0.0),

    # ── Season loss sweep ─────────────────────────────────────────────────────
    dict(name="season_w010", group="sweep_season",
         fft_weight=0.0, trend_weight=0.0, season_weight=0.10),
    dict(name="season_w025", group="sweep_season",
         fft_weight=0.0, trend_weight=0.0, season_weight=0.25),
    dict(name="season_w050", group="sweep_season",
         fft_weight=0.0, trend_weight=0.0, season_weight=0.50),
    dict(name="season_w100", group="sweep_season",
         fft_weight=0.0, trend_weight=0.0, season_weight=1.00),
]

# ── Print plan ────────────────────────────────────────────────────────────────
print(f"Device  : {DEVICE}")
print(f"Total   : {len(EXPERIMENTS)} experiments")
print(f"Protocol: DiffusionTS — full dataset, disc=2000, pred=5000, {N_METRIC_ITERATIONS} runs")
print(f"Eval every {EVAL_METRICS_EVERY} epochs ({1000 // EVAL_METRICS_EVERY} inline checkpoints per run)\n")
print(f"  {'#':<4} {'Name':<14} {'Group':<14} {'fft':>5} {'trend':>6} {'season':>7}")
print("  " + "-" * 52)
_arch = {"raw": "Transformer", "decomposition": "Transformer", "image": "UNet"}
for i, exp in enumerate(EXPERIMENTS):
    print(f"  {i+1:<4} {exp['name']:<14} {exp['group']:<14}"
          f" {exp['fft_weight']:>5.2f} {exp['trend_weight']:>6.2f} {exp['season_weight']:>7.2f}")
print()

# ── Run loop ──────────────────────────────────────────────────────────────────
for i, exp in enumerate(EXPERIMENTS):
    print(f"\n{'='*70}")
    print(f"  [{i+1}/{len(EXPERIMENTS)}]  {exp['name']}  (group: {exp['group']})")
    print(f"{'='*70}\n")

    ckpt_dir  = f"output/ckpt_{exp['name']}"
    best_ckpt = f"{ckpt_dir}/best_model.pt"

    # ── 1. Train ──────────────────────────────────────────────────────────────
    try:
        train(
            mode               = "decomposition",
            device             = DEVICE,
            hidden_dim         = HIDDEN_DIM,
            num_layers         = NUM_LAYERS,
            num_epochs         = NUM_EPOCHS,
            lr                 = LR,
            batch_size         = BATCH_SIZE,
            num_workers        = NUM_WORKERS,
            seed               = SEED,
            use_wandb          = True,
            wandb_project      = WANDB_PROJECT,
            wandb_run_name     = exp["name"],
            wandb_group        = exp["group"],
            eval_metrics       = True,
            eval_metrics_every = EVAL_METRICS_EVERY,
            n_metric_iterations= N_METRIC_ITERATIONS,
            img_pred_objective = "pred_x0",
            img_loss_type      = "l1",
            fft_weight         = exp["fft_weight"],
            trend_weight       = exp["trend_weight"],
            season_weight      = exp["season_weight"],
            checkpoint_dir     = ckpt_dir,
        )
    except Exception:
        import traceback
        print(f"\n!!! TRAINING FAILED: {exp['name']}")
        traceback.print_exc()
        gc.collect(); torch.cuda.empty_cache()
        continue

    gc.collect(); torch.cuda.empty_cache()

    # ── 2. Full post-training evaluation (disc + pred + VDS + FDDS + corr + Context-FID) ─
    if Path(best_ckpt).exists():
        print(f"\n--- Post-training evaluation: {exp['name']} ---")
        try:
            evaluate(
                mode               = "decomposition",
                checkpoint_path    = best_ckpt,
                device             = DEVICE,
                num_samples        = 256,
                n_metric_iterations= N_METRIC_ITERATIONS,
                compute_context_fid= True,
                use_wandb          = True,
                wandb_project      = WANDB_PROJECT,
                wandb_run_name     = exp["name"],
                wandb_group        = exp["group"],
                output_dir         = ckpt_dir,
                hidden_dim         = HIDDEN_DIM,
                num_layers         = NUM_LAYERS,
            )
        except Exception:
            import traceback
            print(f"\n!!! EVALUATION FAILED: {exp['name']}")
            traceback.print_exc()
    else:
        print(f"   [skip eval] best_model.pt not found at {best_ckpt}")

    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*70)
print("  ALL EXPERIMENTS COMPLETE")
print("="*70)

## 3. (Optional) Copy outputs to Drive

Colab runtimes are ephemeral — run this if `REPO_PATH` is not already on Drive.

In [ ]:
import shutil
from pathlib import Path

DRIVE_BACKUP = '/content/drive/MyDrive/TezBaselines/MyCode/output'
LOCAL_OUTPUT = str(Path(REPO_PATH) / 'output')

shutil.copytree(LOCAL_OUTPUT, DRIVE_BACKUP, dirs_exist_ok=True)
print(f"Outputs copied to {DRIVE_BACKUP}")